# Import Fabric Ontology from Lakehouse

Reads an exported ontology definition JSON from a lakehouse Files folder and creates a new ontology in the target workspace.

**What it does:**
1. Reads `*_definition.json` from the configured lakehouse path
2. Validates structure and shows entity/binding summary
3. Rewrites data-source bindings to point at the target lakehouse
4. Creates the ontology (with retry for name conflicts after deletion)
5. Verifies the result

**Prerequisites:**
- Attach the lakehouse which contains the exported `*_definition.json` to the notebook
- Run the **Export** notebook first, then copy the exported `*_definition.json` to the import   
  lakehouse Files folder
- Create a folder 'FabricIQ-export_import_package' under Files section of your lakehouse. 
- Copy 'fabric_ontology_import-1.0.0-py3-none-any.whl' in that folder
- Install the `fabric-ontology-import` wheel in your Fabric environment (or run the `%pip install` 
  cell below)

## 1. Install Package (skip if already in your Fabric Environment)

Uncomment and update the path to your `.whl` file.

In [ ]:
# Uncomment the line below and update the path to your .whl file location
# %pip install /lakehouse/default/Files/FabricIQ-export_import_package/fabric_ontology_import-1.0.0-py3-none-any.whl

## 2. Configuration

**Lakehouse path format:** `abfss://<WorkspaceName>@onelake.dfs.fabric.microsoft.com/<LakehouseName>.Lakehouse/Files`

Copy the exported `*_definition.json` file to the lakehouse Files folder, then set `DEFINITION_PATH` to point at it.

> **Note:** Source item display names are embedded in the definition JSON at export time, so no access to the source workspace is needed.

In [ ]:
# ── Where to find the exported definition ────────────────────
WORKSPACE_NAME      = "<Workspace_name>"
LAKEHOUSE_NAME      = "<Lakehouse_name>"
EXPORT_FOLDER       = "<Export_folder_name>"
SOURCE_ONTOLOGY     = "<Source_ontology_name>"

DEFINITION_PATH = (
    f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_NAME}.Lakehouse/Files/{EXPORT_FOLDER}"
    f"/{SOURCE_ONTOLOGY}_definition.json"
)

# ── Target workspace & new ontology name ─────────────────────
TARGET_WORKSPACE_ID = "<Target_workspace_id>"
NEW_ONTOLOGY_NAME   = "<New_ontology_name>"
DESCRIPTION         = "<Description>"

# ── Target data sources (for binding rewrite) ────────────────
# The ontology's data bindings will be rewritten to point at these items.
# Leave a list empty [] to skip that source type.

#***IMPORTANT***
#Ensure the underlying delta tables within the lakehouses or warehouses and kql tables within the eventhouses/kql databases are 
#created with the same names as the source ontology for bindings to work, else entities will be unbounded
#***IMPORTANT***

TARGET_LAKEHOUSE_NAMES      = ["<Target_lakehouse_name>"]
TARGET_WAREHOUSE_NAMES      = ["<Target_warehouse_name>"]
TARGET_EVENTHOUSE_NAMES     = ["<Target_eventhouse_name>"]
TARGET_SEMANTIC_MODEL_NAMES = ["<Target_semantic_model_name>"]

# ── Behavior ─────────────────────────────────────────────────
OVERWRITE_IF_EXISTS     = True
SKIP_BINDING_VALIDATION = False

print(f"Definition:  {DEFINITION_PATH}")
print(f"Target:      {NEW_ONTOLOGY_NAME} in workspace {TARGET_WORKSPACE_ID}")
print(f"Lakehouses:  {TARGET_LAKEHOUSE_NAMES}")
print(f"Warehouses:  {TARGET_WAREHOUSE_NAMES}")
print(f"Eventhouses: {TARGET_EVENTHOUSE_NAMES}")
print(f"Semantic Models: {TARGET_SEMANTIC_MODEL_NAMES}")

## 3. Import

In [ ]:
from fabric_ontology_import import import_ontology

token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")

result = import_ontology(
    token=token,
    definition_path=DEFINITION_PATH,
    target_workspace_id=TARGET_WORKSPACE_ID,
    new_ontology_name=NEW_ONTOLOGY_NAME,
    description=DESCRIPTION,
    target_lakehouse_names=TARGET_LAKEHOUSE_NAMES,
    target_warehouse_names=TARGET_WAREHOUSE_NAMES,
    target_eventhouse_names=TARGET_EVENTHOUSE_NAMES,
    target_semantic_model_names=TARGET_SEMANTIC_MODEL_NAMES,
    overwrite=OVERWRITE_IF_EXISTS,
    skip_binding_validation=SKIP_BINDING_VALIDATION,
)

print(f"\nOntology ID: {result['ontology_id']}")
print(f"Bindings rewritten: {result['rewrite_count']}")
print(f"Bindings dropped (unbound): {len(result['dropped_bindings'])}")
print(f"Verification: {result['verification']['status']}")